# 04: MLflow, A/B Testing & Model Drift

## Learning Objectives
- Track experiments with MLflow (parameters, metrics, artifacts)
- Register models in MLflow Model Registry
- Design and analyze A/B tests for model deployment
- Detect data and concept drift with Evidently

## Roadmap
MLflow Tracking → Model Registry → A/B Testing → Drift Detection

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
warnings.filterwarnings('ignore')

try:
    import mlflow
    import mlflow.sklearn
    print(f"MLflow: {mlflow.__version__}")
except ImportError:
    print("pip install mlflow")

import sys; sys.path.insert(0, '../../..')
from shared.viz_utils import setup_style
setup_style()

---
## Section 1: MLflow — Experiment Tracking

### Why Track Experiments?
Without tracking:
- "I tried 20 models last week... which one had the best AUC?"
- "What hyperparameters did I use for that Kaggle submission?"
- "I think I overfitted but I don't know what changed..."

With MLflow:
- Every run logged: params, metrics, artifacts, code version
- Compare runs side-by-side in the UI
- Reproducible: know exactly what produced which result

### MLflow Components
| Component | Purpose |
|-----------|---------|
| **Tracking** | Log runs (params, metrics, artifacts) |
| **Projects** | Package code for reproducible runs |
| **Models** | Standard model format |
| **Registry** | Version and stage management (Staging → Production) |

In [ ]:
import mlflow
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import pickle

# Generate churn data
np.random.seed(42); n = 5000
tenure = np.random.exponential(24, n).clip(1, 72).astype(int)
monthly = np.random.normal(65, 25, n).clip(20, 150)
contract = np.random.choice([0, 1, 2], n, p=[0.55, 0.25, 0.20])
calls = np.random.poisson(2, n)
total = tenure * monthly
prob = 1 / (1 + np.exp(-(-2 + 0.05*(monthly-65) - 0.03*tenure + 0.1*calls - 0.5*contract)))
y = (np.random.random(n) < prob).astype(int)
X = pd.DataFrame({'tenure': tenure, 'monthly': monthly, 'total': total,
                  'contract': contract, 'calls': calls})
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Set MLflow tracking URI (local by default)
mlflow.set_tracking_uri("./mlruns")
mlflow.set_experiment("churn-prediction")

def train_and_log(model, model_name, params):
    """Train a model and log everything to MLflow."""
    with mlflow.start_run(run_name=model_name):
        # Log hyperparameters
        mlflow.log_params(params)

        # Train
        start = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - start

        # Evaluate
        y_prob = model.predict_proba(X_val)[:, 1]
        y_pred = model.predict(X_val)

        metrics = {
            "val_auc": roc_auc_score(y_val, y_prob),
            "val_f1": f1_score(y_val, y_pred),
            "val_ap": average_precision_score(y_val, y_prob),
            "train_time_s": round(train_time, 3),
        }
        mlflow.log_metrics(metrics)

        # Log model
        mlflow.sklearn.log_model(model, "model")

        # Log custom artifact (feature importances)
        if hasattr(model, 'feature_importances_'):
            fi = pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_})
            fi.to_csv('/tmp/feature_importances.csv', index=False)
            mlflow.log_artifact('/tmp/feature_importances.csv')

        print(f"  {model_name:35s}: AUC={metrics['val_auc']:.4f}, F1={metrics['val_f1']:.4f}, Time={train_time:.2f}s")
        return metrics

print("Running 3 experiments...\n")

experiments = [
    (GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
     "GBM-baseline", {"n_estimators": 100, "learning_rate": 0.1, "max_depth": 3}),
    (GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
     "GBM-tuned", {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 4}),
    (RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
     "RandomForest", {"n_estimators": 200, "max_depth": 8}),
]

results = {}
for model, name, params in experiments:
    results[name] = train_and_log(model, name, params)

print("\nCompare runs: mlflow ui --backend-store-uri ./mlruns")

In [ ]:
# Query MLflow runs programmatically
try:
    client = mlflow.tracking.MlflowClient(tracking_uri="./mlruns")
    experiment = mlflow.get_experiment_by_name("churn-prediction")
    if experiment:
        runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
        print("All runs:")
        print(runs[['run_id', 'tags.mlflow.runName', 'metrics.val_auc', 'metrics.val_f1',
                     'params.n_estimators', 'params.learning_rate']].to_string(index=False))
except Exception as e:
    print(f"MLflow not running: {e}")
    print("Showing results from memory:")
    for name, m in results.items():
        print(f"  {name}: AUC={m['val_auc']:.4f}, F1={m['val_f1']:.4f}")

---
## Section 2: MLflow Model Registry

### Lifecycle Stages
```
Development → Staging → Production → Archived
```

### Promoting a Model
```python
client = mlflow.tracking.MlflowClient()

# Register model from a run
result = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name="churn-model"
)

# Transition to staging
client.transition_model_version_stage(
    name="churn-model",
    version=result.version,
    stage="Staging",
    archive_existing_versions=False,
)

# After validation, promote to Production
client.transition_model_version_stage(
    name="churn-model",
    version=result.version,
    stage="Production",
    archive_existing_versions=True,  # Archive old Production version
)

# Load the production model anywhere
model = mlflow.sklearn.load_model("models:/churn-model/Production")
```

### Model Registry Workflow
1. All training runs → Tracking Server
2. Best run → Register as model version
3. Team validates in Staging
4. Approve → Promote to Production
5. Previous Production → Archived

In [ ]:
try:
    # Find best run by AUC
    best_run = runs.loc[runs['metrics.val_auc'].idxmax()]
    best_run_id = best_run['run_id']
    best_auc = best_run['metrics.val_auc']

    print(f"Best run: {best_run['tags.mlflow.runName']} (AUC={best_auc:.4f})")

    # Register
    model_uri = f"runs:/{best_run_id}/model"
    registered = mlflow.register_model(model_uri, "churn-model")
    print(f"Registered as version {registered.version}")

    # Transition to staging
    client.transition_model_version_stage(
        name="churn-model", version=registered.version, stage="Staging"
    )
    print("Transitioned to Staging")
except Exception as e:
    print(f"Registry demo (requires MLflow server): {e}")
    print("In production: this is how you promote models through the lifecycle")

---
## Section 3: A/B Testing for Models

### Why A/B Test Models?
- Offline metrics (AUC, F1) don't always predict business impact
- Model A has AUC=0.85, Model B has AUC=0.83 — which one actually generates more revenue?
- A/B tests answer this with statistical confidence

### Design
- **Traffic split**: 50/50 (or 90/10 for riskier changes)
- **Unit of randomization**: Customer ID (consistent experience)
- **Primary metric**: Business KPI (conversion, revenue, churn rate)
- **Duration**: Run until statistical significance (usually 2-4 weeks)
- **Guard rails**: Ensure control metrics don't degrade

### Statistical Testing
- **T-test**: Continuous metrics (revenue, engagement time)
- **Chi-square / Z-test**: Proportions (click rate, conversion rate)
- **Mann-Whitney**: Non-normal distributions (skewed revenue)

In [ ]:
np.random.seed(42)

def simulate_ab_test(n_customers: int, effect_size: float = 0.05):
    """
    Simulate an A/B test between two churn models.

    Control (Model A): baseline model — sends generic retention offers
    Treatment (Model B): new model — sends targeted offers to high-risk customers

    effect_size: expected improvement in retention rate from better targeting
    """
    # Assign customers randomly to A or B
    assignment = np.random.choice(['A', 'B'], size=n_customers)

    # Customer features
    monthly_revenue = np.random.exponential(50, n_customers) + 20
    churn_risk = np.random.beta(2, 5, n_customers)  # True churn probability

    # Model A (baseline): send retention offer to everyone with >30% risk
    model_a_offer = churn_risk > 0.3

    # Model B (new): better calibrated — offers to >35% risk, but more targeted
    # Better model reduces false positives and improves offer acceptance
    model_b_offer = churn_risk > 0.25  # catches more at-risk
    model_b_better_targeting = np.random.random(n_customers) < 0.2  # 20% more effective

    # Simulate whether customer churns (offers reduce churn probability)
    churn_a = np.where(
        (assignment == 'A') & model_a_offer,
        np.random.random(n_customers) < (churn_risk * 0.6),  # 40% reduction with offer
        np.random.random(n_customers) < churn_risk
    )

    churn_b = np.where(
        (assignment == 'B') & model_b_offer,
        np.random.random(n_customers) < (churn_risk * (0.6 - model_b_better_targeting * 0.15)),
        np.random.random(n_customers) < churn_risk
    )

    # Revenue retained
    revenue_a = np.where(assignment == 'A', monthly_revenue * (1 - churn_a), 0)
    revenue_b = np.where(assignment == 'B', monthly_revenue * (1 - churn_b), 0)

    df = pd.DataFrame({
        'assignment': assignment,
        'monthly_revenue': monthly_revenue,
        'churn_risk': churn_risk,
        'churned': np.where(assignment == 'A', churn_a, churn_b),
        'revenue_retained': np.where(assignment == 'A', revenue_a, revenue_b),
    })

    return df

# Run simulation
ab_data = simulate_ab_test(n_customers=10_000, effect_size=0.05)
print("A/B Test Data:")
print(ab_data.groupby('assignment').agg(
    customers=('churned', 'count'),
    churn_rate=('churned', 'mean'),
    avg_revenue=('revenue_retained', 'mean'),
    total_revenue=('revenue_retained', 'sum'),
).round(4))

In [ ]:
# Test for statistical significance
group_a = ab_data[ab_data['assignment'] == 'A']
group_b = ab_data[ab_data['assignment'] == 'B']

# 1. Churn rate difference (proportion test)
n_a, n_b = len(group_a), len(group_b)
churn_a = group_a['churned'].mean()
churn_b = group_b['churned'].mean()
pooled_p = ab_data['churned'].mean()

z_stat = (churn_a - churn_b) / np.sqrt(pooled_p * (1 - pooled_p) * (1/n_a + 1/n_b))
p_value_churn = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print("=== Churn Rate Test ===")
print(f"Model A churn rate: {churn_a:.3f}")
print(f"Model B churn rate: {churn_b:.3f}")
print(f"Difference: {(churn_b - churn_a):.4f} ({(churn_b/churn_a - 1)*100:.1f}%)")
print(f"Z-statistic: {z_stat:.3f}, p-value: {p_value_churn:.4f}")
print(f"Result: {'✓ Significant' if p_value_churn < 0.05 else '✗ Not significant'} (α=0.05)")

# 2. Revenue difference (t-test)
t_stat, p_value_rev = stats.ttest_ind(
    group_a['revenue_retained'],
    group_b['revenue_retained']
)
print("\n=== Revenue Test ===")
print(f"Model A avg revenue: ${group_a['revenue_retained'].mean():.2f}")
print(f"Model B avg revenue: ${group_b['revenue_retained'].mean():.2f}")
print(f"T-statistic: {t_stat:.3f}, p-value: {p_value_rev:.4f}")
print(f"Result: {'✓ Significant' if p_value_rev < 0.05 else '✗ Not significant'} (α=0.05)")

# 3. Confidence intervals
diff_revenue = group_b['revenue_retained'].mean() - group_a['revenue_retained'].mean()
pooled_se = np.sqrt(group_a['revenue_retained'].var()/n_a + group_b['revenue_retained'].var()/n_b)
ci_low = diff_revenue - 1.96 * pooled_se
ci_high = diff_revenue + 1.96 * pooled_se
print(f"\n95% CI for revenue difference: [${ci_low:.2f}, ${ci_high:.2f}]")
annual_impact = diff_revenue * n_customers_per_month if (n_customers_per_month := 50_000) else diff_revenue * 50_000 * 12
print(f"Estimated annual revenue impact (50K customers/month): ${annual_impact:,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Churn rate comparison
churn_by_group = ab_data.groupby('assignment')['churned'].mean()
axes[0].bar(churn_by_group.index, churn_by_group.values, color=['#3498db', '#e74c3c'], alpha=0.8)
axes[0].set_title('Churn Rate by Group', fontsize=13)
axes[0].set_ylabel('Churn Rate')
for i, (idx, val) in enumerate(churn_by_group.items()):
    axes[0].text(i, val + 0.002, f'{val:.3f}', ha='center', fontweight='bold')

# Revenue distribution
for grp, color in [('A', '#3498db'), ('B', '#e74c3c')]:
    rev = ab_data[ab_data['assignment'] == grp]['revenue_retained']
    axes[1].hist(rev[rev > 0], bins=40, alpha=0.5, color=color, label=f'Model {grp}')
axes[1].set_title('Revenue Distribution (Retained Customers)', fontsize=13)
axes[1].set_xlabel('Monthly Revenue ($)')
axes[1].legend()

# Cumulative revenue over time (simulated)
days = np.arange(1, 31)
rev_a_cum = np.cumsum(np.random.normal(group_a['revenue_retained'].mean(), 5, 30))
rev_b_cum = np.cumsum(np.random.normal(group_b['revenue_retained'].mean(), 5, 30))
axes[2].plot(days, rev_a_cum, label='Model A', color='#3498db', linewidth=2)
axes[2].plot(days, rev_b_cum, label='Model B', color='#e74c3c', linewidth=2)
axes[2].fill_between(days, rev_a_cum, rev_b_cum, alpha=0.2, color='green', label='B improvement')
axes[2].set_title('Cumulative Revenue (30 days)', fontsize=13)
axes[2].set_xlabel('Day')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## Section 4: Model Drift Detection

### Types of Drift
| Type | What changes | Example |
|------|-------------|---------|
| **Data drift** | Input distribution | Customer demographics shift |
| **Concept drift** | Relationship between X and y | Pandemic changes churn patterns |
| **Label drift** | Target distribution | Churn rate spikes in recession |
| **Prediction drift** | Model output distribution | Scores becoming more extreme |

### When to Monitor
- After every deployment
- On schedule (daily/weekly)
- When business metrics degrade

### Detection Methods
- **Statistical**: KS test, PSI (Population Stability Index)
- **Distance**: KL divergence, Wasserstein distance
- **Reference windows**: Compare current week vs training week

In [ ]:
# Simulate drift detection without Evidently (for environments without it)
from scipy.stats import ks_2samp

def compute_psi(reference, current, n_bins=10):
    """Population Stability Index — measures distribution shift."""
    bins = np.percentile(reference, np.linspace(0, 100, n_bins + 1))
    bins[0] = -np.inf; bins[-1] = np.inf

    ref_counts = np.histogram(reference, bins=bins)[0] + 0.0001
    cur_counts = np.histogram(current, bins=bins)[0] + 0.0001

    ref_pct = ref_counts / ref_counts.sum()
    cur_pct = cur_counts / cur_counts.sum()

    psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
    return psi

# Simulate reference (training) vs current (production) distributions
np.random.seed(42)
ref_tenure = np.random.exponential(24, 1000).clip(1, 72)
ref_monthly = np.random.normal(65, 25, 1000).clip(20, 150)

# Production data 6 months later — slight drift
current_tenure = np.random.exponential(20, 1000).clip(1, 72)  # Customers leaving faster
current_monthly = np.random.normal(72, 28, 1000).clip(20, 150)  # Prices went up

# KS Test
ks_tenure, p_tenure = ks_2samp(ref_tenure, current_tenure)
ks_monthly, p_monthly = ks_2samp(ref_monthly, current_monthly)

# PSI
psi_tenure = compute_psi(ref_tenure, current_tenure)
psi_monthly = compute_psi(ref_monthly, current_monthly)

print("Drift Detection Report")
print("="*50)
print(f"\nFeature: tenure_months")
print(f"  KS statistic: {ks_tenure:.4f}, p-value: {p_tenure:.6f}")
print(f"  PSI: {psi_tenure:.4f} {'🔴 HIGH DRIFT' if psi_tenure > 0.2 else '🟡 MODERATE' if psi_tenure > 0.1 else '🟢 OK'}")
print(f"  Ref mean: {ref_tenure.mean():.1f} → Current mean: {current_tenure.mean():.1f}")

print(f"\nFeature: monthly_charges")
print(f"  KS statistic: {ks_monthly:.4f}, p-value: {p_monthly:.6f}")
print(f"  PSI: {psi_monthly:.4f} {'🔴 HIGH DRIFT' if psi_monthly > 0.2 else '🟡 MODERATE' if psi_monthly > 0.1 else '🟢 OK'}")
print(f"  Ref mean: {ref_monthly.mean():.1f} → Current mean: {current_monthly.mean():.1f}")

# PSI interpretation
print("\nPSI Thresholds:")
print("  < 0.10: No significant drift")
print("  0.10–0.20: Moderate drift — investigate")
print("  > 0.20: Significant drift — retrain model")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (ref, cur, name, unit) in zip(axes, [
    (ref_tenure, current_tenure, 'Tenure', 'months'),
    (ref_monthly, current_monthly, 'Monthly Charges', '$'),
]):
    ax.hist(ref, bins=30, alpha=0.5, label='Training (reference)', color='#3498db', density=True)
    ax.hist(cur, bins=30, alpha=0.5, label='Production (current)', color='#e74c3c', density=True)
    ax.set_title(f'{name} Distribution Drift', fontsize=13)
    ax.set_xlabel(f'{name} ({unit})')
    ax.set_ylabel('Density')
    ax.legend()

plt.tight_layout()
plt.show()

---
## Key Takeaways

1. **MLflow** = track EVERYTHING — params, metrics, artifacts, git hash
2. **Model Registry** = formalize the Staging → Production lifecycle
3. **A/B tests** = the only way to measure true business impact of model changes
4. **Statistical significance** = p < 0.05 AND practical significance (effect size matters)
5. **Drift** = models degrade silently — monitor distributions weekly with PSI/KS test
6. **PSI > 0.2** = retrain your model

---
## Exercises

### Exercise 1: MLflow Experiment
Run 5 different hyperparameter combinations for GBM, all in the same MLflow experiment. Use `mlflow.search_runs()` to find the best and worst run by AUC.

### Exercise 2: Sample Size Calculator
Write a function `ab_sample_size(baseline_rate, min_detectable_effect, alpha=0.05, power=0.8)` that returns the required sample size per group for an A/B test.

### Exercise 3: Drift Alert
Build a function `drift_report(ref_df, current_df)` that:
- Computes PSI for every numeric column
- Flags columns with PSI > 0.1
- Returns a DataFrame with column name, PSI value, and status